# 4 - Build Analysis-Ready Datasets

**Purpose**: Merge CIDER metrics with metadata, COBALT sequencing depths, and AMBER depths.
Compute CIDER Raw Score and CIDER Score (purity + depth normalised).
Assign broad cohort factors, High/Low infiltration groups, and CIDER × TMB groups.

| | |
|---|---|
| **Inputs** | `CIDER_PROJ_DIR/cider_base.csv`, `meta_base.csv`, `COBALT_DIR/depths.csv`, `AMBER_DIR/amber_depths.csv` |
| **Outputs** | `CIDER_PROJ_DIR/ready.rds`, `cpi_ready.rds` |
| **Run after** | `1_cider.ipynb`, `3_meta.ipynb` |
| **Run before** | `5_models.ipynb` |

In [1]:
source(paste0(dirname(getwd()), "/helpers/map.r"))
source(paste0(dirname(getwd()), "/helpers/shortcuts.r"))
source(paste0(dirname(getwd()), "/helpers/config.r"))

assert_files_exist(
  "CIDER base"  = paste0(CIDER_PROJ_DIR, "cider_base.csv"),
  "Meta base"     = paste0(CIDER_PROJ_DIR, "meta_base.csv"),
  "COBALT depths" = paste0(COBALT_DIR, "depths.csv"),
  "AMBER depths"  = paste0(AMBER_DIR,  "amber_depths.csv")
)

In [2]:
# purrr and broom are not auto-attached by shortcuts.r
library(purrr)
library(broom)

# 0 - Read prepared CIDER and Metadata

In [3]:
cider_base <- fread(paste0(CIDER_PROJ_DIR, "cider_base.csv"))

In [4]:
meta_base  <- fread(paste0(CIDER_PROJ_DIR, "meta_base.csv"))

# 1 - Combine CIDER and metadata

In [5]:
depths       <- fread(paste0(COBALT_DIR, "depths.csv"))
amber_depths <- fread(paste0(AMBER_DIR,  "amber_depths.csv"))

In [6]:
# left_join preserves all samples from meta_base, including those with
# no CDR3 sequences detected (zero immune infiltration). These are biologically
# meaningful and should not be silently dropped.
# CIDER count NAs (samples absent from cider_base) are filled with 0.
# Depth NAs (samples absent from depths) produce NA CIDER scores downstream.
# Zero-fill for true zero-infiltration samples is handled upstream in
# 0_cider.ipynb using the intersection of clinical.txt and purity_ready.csv.
# Samples with NA gp here have no WGS or no clinical data — exclude them.
base <- meta_base |>
  left_join(cider_base, by = "sampleId") |>
  filter(!is.na(gp)) |>
  left_join(depths,       by = "sampleId") |>
  left_join(amber_depths, by = "sampleId")

# Sanity check: (sampleId, gp) must be unique — duplicates indicate a join error
n_dupes <- base |> group_by(sampleId, gp) |> filter(n() > 1) |> nrow()
if (n_dupes > 0) {
  stop(sprintf("Duplicate (sampleId, gp) rows detected: %d rows. Check joins.", n_dupes),
       call. = FALSE)
}
message("Deduplication check passed: ", nrow(base), " rows, ",
        n_distinct(base$sampleId), " unique samples × ",
        n_distinct(base$gp), " groups")

In [7]:
# Four CDR3 score variants:
#
#   cider_score            — cdr3_unique                    raw unique count (integer)
#   cider_score_depth_adj  — (cdr3_unique / sampleMedianDepth) × SCORE_SCALE
#                            COBALT depth-normalised only; used for RNA and ImmuneLens validation
#   cider_score_adj        — cdr3_unique / (sampleMedianDepth * max(1-purity, PURITY_CAP))
#                            primary clinical metric: COBALT depth + purity corrected
#   cider_total            — cdr3_total                     raw total CDR3 reads
#   cider_total_depth_adj  — (cdr3_total / sampleMedianDepth) × SCORE_SCALE
#                            depth-normalised total reads; used for unique vs total comparison
#
# COBALT genome-wide median depth is used for normalisation as it represents
# overall sequencing effort across the whole genome.
# No log2 transformation — scores are on an interpretable ratio scale.
# log1p() is applied before scaling in 5_models.ipynb.
# PURITY_CAP (0.10) assumes at least 10% non-tumour fraction.

base <- base |>
  mutate(
    cider_score           = cdr3_unique,
    cider_score_depth_adj = (cdr3_unique / sampleMedianDepth) * SCORE_SCALE,
    cider_score_adj       = (cdr3_unique / (sampleMedianDepth * pmax(1 - purity, PURITY_CAP))) * SCORE_SCALE,
    cider_total           = cdr3_total,
    cider_total_depth_adj = (cdr3_total / sampleMedianDepth) * SCORE_SCALE
  )

In [8]:
ready <- bind_rows(
  base,
  base |> mutate(cohort_broad = "ICI-Treated")
)

# 2 - CPI patient dataset

- Build cohort levels

In [9]:
# CPI cohort levels for clinical analyses (Figures 3–5)
# Defined in helpers/config.r

# Pan cohort levels for full-database validation (Figure 2A)
# Defined in helpers/config.r

# ── Full-database ready object (all patients) — for Figure 2 validation ───────
ready <- ready |>
  group_by(cohort_pan) |>
  mutate(n_pan = n_distinct(sampleId)) |>
  ungroup() |>
  mutate(
    cohort_pan    = factor(cohort_pan, levels = PAN_LEVELS),
    cohort_pan_go = {
      lkp <- distinct(pick(cohort_pan, n_pan)) |> arrange(cohort_pan)
      factor(
        paste0(cohort_pan, " (N = ", n_pan, ")"),
        levels = paste0(lkp$cohort_pan, " (N = ", lkp$n_pan, ")")
      )
    }
  ) |>
  select(-n_pan)

# ── CPI-only subset — for all clinical analyses ───────────────────────────────
cpi_ready <-
  ready |>
  filter(cpi) |>
  drop_na(response, osDays, osEvent) |>
  group_by(cohort_broad) |>
  mutate(n_broad = n_distinct(sampleId)) |>
  ungroup() |>
  mutate(
    cohort_broad    = factor(cohort_broad, levels = BROAD_LEVELS),
    cohort_broad_go = {
      lkp <- distinct(pick(cohort_broad, n_broad)) |> arrange(cohort_broad)
      factor(
        paste0(cohort_broad, " (N = ", n_broad, ")"),
        levels = paste0(lkp$cohort_broad, " (N = ", lkp$n_broad, ")")
      )
    }
  ) |>
  select(-n_broad)

- Add factor levels for lymphocyte type

In [10]:
# gp_levels defined as GP_LEVELS in helpers/config.r

cpi_ready <- cpi_ready |> mutate(gp = factor(gp, levels = GP_LEVELS))

- High/Low lymphocyte groups

In [11]:
cpi_ready <-
  cpi_ready |>
  group_by(gp) |>
  mutate(
    high_low      = ntile(cider_score_depth_adj, 2),
    group         = factor(
      case_when(high_low == 1 ~ "Low", high_low == 2 ~ "High"),
      levels = c("Low", "High")
    )
  ) |>
  ungroup()

- Combined lymphocyte + TMB groups

In [12]:
cpi_ready <-
  cpi_ready |>
  mutate(combined_group = factor(
    case_when(
      group == "Low"  & tmb_high == 0 ~ paste0(gp, " Low / TMB Low"),
      group == "Low"  & tmb_high == 1 ~ paste0(gp, " Low / TMB High"),
      group == "High" & tmb_high == 0 ~ paste0(gp, " High / TMB Low"),
      group == "High" & tmb_high == 1 ~ paste0(gp, " High / TMB High")
    ),
    levels = c(
      paste0(GP_LEVELS, " Low / TMB Low"),
      paste0(GP_LEVELS, " Low / TMB High"),
      paste0(GP_LEVELS, " High / TMB Low"),
      paste0(GP_LEVELS, " High / TMB High")
    )
  ))

# 3 - Send it

In [13]:
saveRDS(ready, paste0(CIDER_PROJ_DIR, "ready.rds"))
saveRDS(cpi_ready, paste0(CIDER_PROJ_DIR, "cpi_ready.rds"))